# Portfolio website: local development and Cloudflare deployment

This notebook is a practical runbook for the `portfolio-website` repository. It covers:

1. running the site locally with hot reload;
2. testing the production build at localhost;
3. pushing the repository to GitHub; and
4. connecting GitHub to Cloudflare Workers for automatic deployments.

> Run terminal commands from the repository root. Cells that authenticate, push, or deploy are intentionally shown as examples and should only be run when you are ready.

## 1. Prerequisites

Install these tools first:

- Git
- Node.js 22.12 or newer
- npm
- an optional notebook viewer such as VS Code's Jupyter extension
- a GitHub account and Cloudflare account for deployment

This project uses vinext: a Vite-based implementation of the Next.js API. Node 18 is too old for the current build toolchain.

In [ ]:
# Check the tools available in your terminal
!node --version
!npm --version
!git --version

If `node --version` is below `v22.12.0`, switch versions with your preferred Node version manager. With nvm:

```bash
nvm install 22
nvm use 22
node --version
```

The repository's `package.json` also declares the required Node version.

## 2. Install the project

Clone the GitHub repository if you are starting on another computer. Otherwise, open a terminal in the existing local repository.

```bash
git clone https://github.com/j-watterson/portfolio-website.git
cd portfolio-website
npm ci
```

`npm ci` installs the exact dependency versions recorded in `package-lock.json`. Use `npm install` when intentionally changing dependencies.

In [ ]:
# Run this from the repository root
!npm ci

## 3. Run the development server

Start the development server:

```bash
npm run dev
```

Open **http://localhost:3000** in a browser. Keep the terminal running while you edit files. Changes should appear automatically through hot reload.

Stop the server with `Ctrl+C`.

If port 3000 is already occupied, choose another port:

```bash
npm run dev -- --port 3001
```

### Where to edit content

- `app/page.tsx` — homepage
- `lib/projects.ts` — shared project content and case-study data
- `app/about/page.tsx` — About page
- `app/resume/page.tsx` — résumé page
- `app/globals.css` — global visual design
- `public/` — static assets

The project routes are generated from `lib/projects.ts`, so update that file instead of duplicating project details across pages.

## 4. Test the production build locally

A development server is forgiving. Before every push, also test the optimized production build:

```bash
npm run check
npm run build
npm run start
```

Then open **http://localhost:3000** again. `npm run build` creates the server bundle in `dist/`; `npm run start` serves that bundle.

A useful pre-push check is:

```bash
npm audit --audit-level=high
git status
```

In [ ]:
# Safe validation commands; these wrappers use Node 22 even if the host has Node 18
!npm run check:node22
!npm run build:node22
!npm audit --audit-level=high

## 5. Push your work to GitHub

Inspect what will be committed, commit it, and push it:

```bash
git status
git add .
git commit -m "Update portfolio website"
git push origin main
```

If GitHub asks for authentication, use GitHub CLI (`gh auth login`) or a credential manager. GitHub no longer accepts an account password for Git operations over HTTPS.

## 6. Prepare vinext for Cloudflare Workers

Create a branch before the one-time Cloudflare migration so you can review its generated configuration:

```bash
git switch -c configure-cloudflare
npx vinext init --platform=cloudflare
```

The initializer adds Cloudflare's Vite integration and creates or updates `wrangler.jsonc`. Review the diff:

```bash
git diff
npm run check
npm run build
```

Important checks:

- the Worker `name` in `wrangler.jsonc` should be unique in your Cloudflare account;
- keep the generated `compatibility_flags`, including `nodejs_compat` when present;
- do not commit API tokens or `.env` files;
- commit `wrangler.jsonc` and the Cloudflare-aware `vite.config.ts` after validation.

This repository also copies `.openai/hosting.json` into its local `dist/` bundle for OpenAI Sites. That metadata is unrelated to Cloudflare and does not contain a deployment secret.

## 7. Optional first deployment from your computer

A local first deployment is useful because it verifies Cloudflare authentication and configuration before enabling Git-based builds.

```bash
npx wrangler login
npx wrangler whoami
npx @vinext/cloudflare deploy
```

`wrangler login` opens a browser and stores local credentials. For CI, use a scoped `CLOUDFLARE_API_TOKEN` secret instead—never commit a token.

After deployment, Wrangler prints the `workers.dev` URL. Visit it and check the homepage and all three project routes.

## 8. Connect GitHub to Cloudflare

In the Cloudflare dashboard:

1. Open **Workers & Pages** and choose **Create / Import a repository**.
2. Authorize GitHub and select `j-watterson/portfolio-website`.
3. Select `main` as the production branch.
4. Leave the root directory as `/` because the app is at the repository root.
5. Make sure the Cloudflare Worker name matches the `name` in `wrangler.jsonc`.
6. Use `npm ci` as the dependency-install command if Cloudflare exposes that setting.
7. Use `npm run build` as the build command.
8. Use `npx wrangler deploy` as the deploy command.
9. Save the configuration and trigger the first build.

Cloudflare Workers Builds will then build and deploy commits pushed to `main`. Other branches can create preview versions, depending on the preview-build settings in the dashboard.

If you prefer vinext's combined command, leave the separate build command empty and use `npx @vinext/cloudflare deploy` as the deploy command. Do not configure both a full build and a second full vinext build unless you intentionally accept the duplicate work.

## 9. Recommended everyday workflow

```text
Edit locally
   ↓
npm run dev
   ↓
npm run check && npm run build
   ↓
Commit and push a feature branch
   ↓
Review Cloudflare preview
   ↓
Merge to main
   ↓
Cloudflare deploys production
```

Use pull requests even when working alone. They provide a clean diff, a Cloudflare preview, and a useful change history.

## 10. Troubleshooting

**`npm` reports an unsupported Node engine**  
Switch to Node 22.12 or newer, delete no files, and rerun `npm ci`.

**localhost does not open**  
Confirm the terminal still shows a running server. Try `npm run dev -- --port 3001`, then open `http://localhost:3001`.

**Cloudflare says the Worker name does not match**  
Make the dashboard Worker name and `wrangler.jsonc` `name` identical.

**Cloudflare builds twice**  
Use either the separate `npm run build` + `npx wrangler deploy` flow or the combined `npx @vinext/cloudflare deploy` flow.

**A GitHub push is rejected**  
Run `git pull --rebase origin main`, resolve any conflicts, retest, and push again. Never use a destructive reset to bypass work you have not reviewed.

**A deployed route returns an error**  
Check the Cloudflare build log first, then use `npx wrangler tail` while requesting the failing route.

## Official references

- vinext repository and deployment guide: https://github.com/cloudflare/vinext
- Cloudflare Workers Builds: https://developers.cloudflare.com/workers/ci-cd/builds/
- Workers Builds configuration: https://developers.cloudflare.com/workers/ci-cd/builds/configuration/
- Import an existing Git project: https://developers.cloudflare.com/workers/framework-guides/automatic-configuration/

These tools evolve quickly. Recheck the official documentation if dashboard labels or generated configuration differ from the screenshots or instructions you see elsewhere.